# Data Loading — FastF1

**Goal:** pull historical F1 race results (2018–2026) into one tidy DataFrame and cache it to `data/raw/` for the features/model notebooks.

**Approach:** learn the FastF1 API on a single race first, then generalize to a season/round loop.

In [ ]:
import fastf1
import pandas as pd
import numpy as np

from f1pred import paths

# FastF1 caches raw API responses to disk -> huge speedup on re-runs.
CACHE_DIR = paths.DATA / "cache" / "fastf1"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))

paths.ensure_dirs()
print("cache:", CACHE_DIR)

## 1. Load one session (learn the API)

Start with a single known race and inspect what FastF1 returns before scaling up.

In [ ]:
# One race: 2024 season, round 1 (Bahrain), Race session = "R".
session = fastf1.get_session(2024, 1, "R")
session.load()   # first run downloads (slow); afterwards it's cached

# Things to inspect:
#   session.results  -> DataFrame, one row per driver
#                       (DriverId, TeamId, GridPosition, Position, Points, Status...)
#   session.event    -> event metadata (EventName, RoundNumber, EventDate...)
results = session.results
results.head()

## 2. Extract the columns you care about

Reduce the wide results table to a tidy *row-per-driver-per-race* frame.

In [ ]:
def tidy_results(session) -> pd.DataFrame:
    """Return one tidy row per driver for a loaded session."""
    res = session.results
    ev = session.event

    # TODO choose columns. Candidates in session.results:
    #   DriverId, Abbreviation, TeamId, TeamName,
    #   GridPosition, Position, Points, Status
    cols = [
        # "DriverId", "Abbreviation", "TeamId",
        # "GridPosition", "Position", "Points", "Status",
    ]
    df = res[cols].copy()

    # TODO add race identifiers so rows stay self-describing after concat:
    # df["season"] = ev["EventDate"].year
    # df["round"]  = ev["RoundNumber"]
    # df["event"]  = ev["EventName"]

    return df


# tidy_results(session).head()

## 3. Loop over seasons and rounds

Once one race works, generalize. Wrap each load in `try/except` — future or not-yet-run rounds will fail, and that's expected.

In [ ]:
SEASONS = range(2018, 2027)   # 2018 ... 2026

frames = []
for season in SEASONS:
    schedule = fastf1.get_event_schedule(season)   # all rounds in the season
    for rnd in schedule["RoundNumber"]:
        if rnd == 0:
            continue   # round 0 = pre-season testing
        try:
            s = fastf1.get_session(season, int(rnd), "R")
            s.load()
            frames.append(tidy_results(s))
            print(f"ok   {season} R{rnd}")
        except Exception as e:
            print(f"skip {season} R{rnd}: {e}")

# races = pd.concat(frames, ignore_index=True)
# races.shape

## 4. Save to `data/raw/`

Persist as parquet so the features/model notebooks just read it back.

In [ ]:
# out = paths.RAW / "races.parquet"
# races.to_parquet(out, index=False)
# print("wrote", out, races.shape)